In [3]:
import pandas as pd 
import random
import re
import geopandas as gpd

import requests



In [4]:
df_ref = pd.read_csv("../../data/ecoles/fr-en-adresse-et-geolocalisation-etablissements-premier-et-second-degre.csv", sep=";")
df_ref = df_ref.dropna(subset='adresse_uai')

df_pat = pd.read_csv("../../data/data_cleaned/patients_FR_geocoded_adulte_clinique.csv", sep=";",dtype =str)

df_pat = df_pat.dropna(subset='adresse')

df_pat = df_pat[['pseudo_provisoire','requete','adresse','codepost','nom_commune_postal']]

In [5]:
df_revenu = pd.read_csv("../../data/insee_revenu/BASE_TD_FILO_DISP_IRIS_2020.csv",sep=";",dtype=str)[["IRIS","DISP_MED20"]]
df_iris = gpd.read_file('../../data/zones_geographiques/iris/CONTOURS-IRIS.shp')


In [6]:
## Ajout de l'information du revenu associé à l'iris :

gdf = (gpd.GeoDataFrame(df_ref, geometry=gpd.points_from_xy(df_ref.longitude, df_ref.latitude)).set_crs(epsg=4326)).to_crs(epsg=2154)

df_ref_iris = gpd.sjoin(gdf, df_iris[['CODE_IRIS','geometry']], how="left", op='within')
df_ref_iris.drop('index_right', axis=1, inplace=True)

df_ref_iris = df_ref_iris[~ pd.isna(df_ref_iris['CODE_IRIS'])]

df_ref_rev = df_ref_iris.merge(df_revenu, left_on='CODE_IRIS',right_on="IRIS")

df_ref_rev = df_ref_rev.drop('IRIS',axis=1).rename({'CODE_IRIS':'CODE_IRIS_ini','DISP_MED20':'DISP_MED20_ini'},axis=1)


/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3361: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if (await self.run_code(code, result,  async_=asy)):


In [7]:
def add_bruit_unique(df, biais, pos_bruit,add_elem=False):

    df_bruite = df.copy() 

    for i in df_bruite.index:
        adresse = df_bruite.loc[i,'adresse'] 

        if pos_bruit =='AV':
            if add_elem == True  :
                j = random.randint(0,3)
                add = liste_nom_propre[j]
                adresse_biais = biais + ' ' + add  + ' ' +  adresse 
                df_bruite.loc[i,'adresse'] = adresse_biais
            else : 
                adresse_biais = biais + ' ' +  adresse 
                df_bruite.loc[i,'adresse'] = adresse_biais
        if pos_bruit =='AP':
            if add_elem == True  :
                j = random.randint(0,3)
                add = liste_nom_propre[j]
                adresse_biais = adresse + ' ' + biais + ' ' + add 
                df_bruite.loc[i,'adresse'] = adresse_biais
            else : 
                adresse_biais = adresse + ' ' + biais  
                df_bruite.loc[i,'adresse'] = adresse_biais
            

    return df_bruite


In [13]:
%run cleaning_functions.py

In [16]:
cols_to_keep = ['numero_uai','adresse_uai','code_postal_uai','libelle_commune','longitude','latitude','code_departement','code_region','CODE_IRIS_ini', 'DISP_MED20_ini']

liste_biais = [["RESIDENCE",'AV'],["CHEZ",'AV'],["BAT",'AP'],["APPT",'AP'],["MME",'AV'],["MR",'AV'],["RES",'AP'],["HOPITAL",'AV'],["MAISON",'AV'],["RETRAITE",'AV'],["CENTRE",'AV'],["HOTEL",'AV'],["QUARTIER",'AP']]
liste_nom_propre = ["CHARLES DE GAULLES", "JEAN MOULIN", "MARIE CURIE", "FOCH"]



df_ref = df_ref_rev[cols_to_keep]

reg_metrop = ['11','24','27','28','32','44','54','53','75','76','84','93','94']

df_ref["code_region"] = df_ref["code_region"].astype(str)
df_ref_metrop = df_ref[df_ref["code_region"].isin(reg_metrop)]

df_ref.rename(columns={'adresse_uai': 'adresse'}, inplace=True)

df_ref['adresse'] = df_ref['adresse'].astype(str).apply(lambda x: x.upper())
df_ref = prep_adresse(df_ref)


/tmp/ipykernel_17885/242805378.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ref["code_region"] = df_ref["code_region"].astype(str)
/tmp/ipykernel_17885/242805378.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ref.rename(columns={'adresse_uai': 'adresse'}, inplace=True)
/tmp/ipykernel_17885/242805378.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guid

In [18]:
df_ref_biais = {}

for biais, pos_bruit in liste_biais:
    df_ref_rand = df_ref.sample(n=1000).reset_index()

    if biais in ['RESIDENCE','RES','CHEZ','MR','MME','HOTEL']: 
        df_ref_biais[f"df_ref_{biais}_np"] = add_bruit_unique(df_ref_rand,biais,pos_bruit, True)
        df_ref_biais[f"df_ref_{biais}"] = add_bruit_unique(df_ref_rand,biais,pos_bruit)

    else :
        df_ref_biais[f"df_ref_{biais}"] = add_bruit_unique(df_ref_rand,biais,pos_bruit)


def ref_to_geocoding(df):
    for i in df.index :
        df.loc[i,"requete"] =df.loc[i,'adresse'] + " " + str(df.loc[i,"code_postal_uai"])+ " " + df.loc[i,"libelle_commune"]
    return df

for nom_df, df in df_ref_biais.items():
    df_modifie = ref_to_geocoding(df)
    df_ref_biais[nom_df] = df_modifie
    print(f"DataFrame: {nom_df} prêt à être géocodé")
    print("---")



DataFrame: df_ref_RESIDENCE_np prêt à être géocodé
---
DataFrame: df_ref_RESIDENCE prêt à être géocodé
---
DataFrame: df_ref_CHEZ_np prêt à être géocodé
---
DataFrame: df_ref_CHEZ prêt à être géocodé
---
DataFrame: df_ref_BAT prêt à être géocodé
---
DataFrame: df_ref_APPT prêt à être géocodé
---
DataFrame: df_ref_MME_np prêt à être géocodé
---
DataFrame: df_ref_MME prêt à être géocodé
---
DataFrame: df_ref_MR_np prêt à être géocodé
---
DataFrame: df_ref_MR prêt à être géocodé
---
DataFrame: df_ref_RES_np prêt à être géocodé
---
DataFrame: df_ref_RES prêt à être géocodé
---
DataFrame: df_ref_HOPITAL prêt à être géocodé
---
DataFrame: df_ref_MAISON prêt à être géocodé
---
DataFrame: df_ref_RETRAITE prêt à être géocodé
---
DataFrame: df_ref_CENTRE prêt à être géocodé
---
DataFrame: df_ref_HOTEL_np prêt à être géocodé
---
DataFrame: df_ref_HOTEL prêt à être géocodé
---
DataFrame: df_ref_QUARTIER prêt à être géocodé
---


In [19]:
def geocode(df,liste_biais, join = False):
    """Geocode dataframe with Etalab addok"""
    name =[x for x in globals() if globals()[x] is df][0]
    print(f"Proceed to geocode on address for {name}...")
    
    for i in df.index:
        try:
            # get json response
            r = requests.get('https://addok-data.curie.net/search?q='+df["requete"][i])
            response = r.json()

            if i%10000==0 : 
                print(f"Proceed geocoding at the {i}th row")
            if response["features"]!=[]:
                # parse json to insert value in dataframe
                df.at[i, 'x'] = str(response["features"][0]["geometry"]["coordinates"][0])
                df.at[i, 'y'] = str(response["features"][0]["geometry"]["coordinates"][1])
                df.at[i, 'score'] = str(response["features"][0]["properties"]["score"])

                if float(df["score"][i])<0.4:
                    df.at[i, 'trust_score'] = 'low'
                elif float(df["score"][i])>0.4 and float(df["score"][i])<0.65:
                    df.at[i, 'trust_score'] = 'middle'
                elif float(df["score"][i])>0.65 and float(df["score"][i])<0.9:
                    df.at[i, 'trust_score'] = 'middle'
                else:
                    df.at[i, 'trust_score'] = 'high'

                df.at[i, 'street'] = str(response["features"][0]["properties"]["name"]).replace("'", " ")
                df.at[i, 'city'] = str(response["features"][0]["properties"]["city"]).replace("'", " ")
                df.at[i, 'pc_city'] = str(response["features"][0]["properties"]["postcode"])
                df.at[i, 'ic_city'] = str(response["features"][0]["properties"]["citycode"])

                context = (str(response["features"][0]["properties"]["context"]).replace("'", " ")).split(",")
                df.at[i, 'code_dept'] = context[0]
                df.at[i, 'dept'] = context[1]

                # if len(df["code_dept"][i])==2:
                #     df.at[i, 'reg'] = context[2]
                # else:
                #     df.at[i, 'reg'] = "other"

                df.at[i, "address"] = str(response["features"][0]["properties"]["label"]).replace("'", " ")
                
#                 if (df["adresse"][i].lstrip())[0].isdigit():
#                     df.at[i, 'address_has_num_init'] = "true"
#                 else:
#                     df.at[i, 'address_has_num_init'] = "false"
                    
#                 if (df["street"][i].lstrip())[0].isdigit():
#                     df.at[i, 'address_has_num_geoloc'] = "true"
#                 else:
#                     df.at[i, 'address_has_num_geoloc'] = "false"
                    
                if df["codepost"][i] == df["pc_city"][i]:
                    df.at[i, 'same_city'] = "true"
                else:
                    df.at[i, 'same_city'] = "false" 

                # ##Analyse biais : 
                # for mot in liste_biais: 
                #     if mot in df["adresse"].str.split(' '):
                #         df.at[i,f"adresse_has_{mot}_init"] = True 
                #     else : 
                #         df.at[i,f"adresse_has_{mot}_init"] = False 

                    
                ## Check up si mot "hotel" et "chez" sont présent dans la réponse du géocodage?     
                # if "hotel" in df["street"][i] or "hôtel" in df["street"][i]: 
                #     df.at[i, 'hostel'] = "true"
                # else:
                #     df.at[i, 'hostel'] = "false"

                # if "chez" in df["street"][i]: 
                #     df.at[i, 'hosted'] = "true"
                # else:
                #     df.at[i, 'hosted'] = "false"
                    
                df.at[i, 'date_geoloc'] = str(datetime.date.today())
                #df.at[i, 'etalab_version'] = str(response["licence"])
                #df.at[i, 'ban_version'] = "2021-04-27"
            else:
                pass
        except:
            pass
    return df


In [20]:
def spatialjoin(df, df_iris, df_epci, df_dept):
    """Spatial join of the database and the iris/epci layers"""
    name =[x for x in globals() if globals()[x] is df][0]
    print(f"Perfom to spatial join on IRIS and EPCI layers for {name}...")
    gdf = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y)).set_crs(epsg=4326)).to_crs(epsg=2154)

    df_join_iris = gpd.sjoin(gdf, df_iris[['CODE_IRIS','geometry']], how="left", op='within')
    df_join_iris.drop('index_right', axis=1, inplace=True)

    df_epci =df_epci.to_crs(epsg=2154)
    df_join_iris_epci = gpd.sjoin(df_join_iris, df_epci[['CODE_EPCI','geometry']], how="left", op='within')
    df_join_iris_epci.drop('index_right', axis=1, inplace=True)

    df_join_iris_epci_dept = gpd.sjoin(df_join_iris_epci, df_dept[['CODE_DEPT','geometry']], how="left", op='within')
    df_join_iris_epci_dept.drop('index_right', axis=1, inplace=True)
    return df_join_iris_epci_dept


# df_iris = gpd.read_file('../../data/zones_geographiques/IRIS/CONTOURS-IRIS.shp')
df_epci = gpd.read_file('../../data/zones_geographiques/EPCI/EPCI_SHAPEFILE.shp')
df_dept = gpd.read_file('../../data/zones_geographiques/DEPT/DEPARTEMENT.shp')

In [23]:
df_ref_biais_geocoded = {}

for nom_df, df in df_ref_biais.items():
    df_geocoded = geocode(df,liste_biais)
    print(f"{nom_df} géocodé")
    df_geo_spatial = spatialjoin(df_geocoded, df_iris, df_epci, df_dept)
    df_ref_biais_geocoded[nom_df] = df_geo_spatial
    df_geo_spatial.to_csv(f"../../data/data_biaised/{nom_df}.csv",sep=";")
    print(f"{nom_df} jointure spatiale faite")
    print("---")

Proceed to geocode on address for df...
Proceed geocoding at the 0th row
df_ref_RESIDENCE_np géocodé
Perfom to spatial join on IRIS and EPCI layers for df...


/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


df_ref_RESIDENCE_np jointure spatiale faite
---
Proceed to geocode on address for df...
Proceed geocoding at the 0th row
df_ref_RESIDENCE géocodé
Perfom to spatial join on IRIS and EPCI layers for df...


/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


df_ref_RESIDENCE jointure spatiale faite
---
Proceed to geocode on address for df...
Proceed geocoding at the 0th row
df_ref_CHEZ_np géocodé
Perfom to spatial join on IRIS and EPCI layers for df...


...

/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


df_ref_HOTEL jointure spatiale faite
---
Proceed to geocode on address for df...
Proceed geocoding at the 0th row
df_ref_QUARTIER géocodé
Perfom to spatial join on IRIS and EPCI layers for df...


/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


df_ref_QUARTIER jointure spatiale faite
---


/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [12]:
def word_in_adresse(df,mot):
    return df[df['adresse'].str.contains(mot,case=False,na=False)]


word_in_adresse(df_pat,' QUARTIER ')

,Unnamed: 0,geocancer_id,requete,adresse,cpville,ville,voirie,numeros,bruit,pos_voirie,pos_numeros,pos_bruit,pos_prc_voirie,pos_prc_numeros,pos_prc_bruit,elem_adresse,elem_bruit,bruit_AV_AP
531,532,geoc_007158,0 QUARTIER PLAISANCE 97280 LE VAUCLIN,0 QUARTIER PLAISANCE,97280.0,LE VAUCLIN,QUARTIER,0,NaN,2.0,1.0,NaN,66.67,33.33,NaN,PLAISANCE,NaN,NaN
2150,2151,geoc_001504,RUE DES NOISETIERS QUARTIER ANFA 99350 CASABLANCA,RUE DES NOISETIERS QUARTIER ANFA,99350.0,CASABLANCA,RUE,NaN,QUARTIER,1.0,NaN,4,20.0,NaN,80.0,DES NOISETIERS,QUARTIER ANFA,AP
4445,4447,geoc_012526,00 QUARTIER ELMONDAZAH 99201 BURAYDAH,00 QUARTIER ELMONDAZAH,99201.0,BURAYDAH,QUARTIER,00,NaN,2.0,1.0,NaN,66.67,33.33,NaN,ELMONDAZAH,NaN,NaN
9682,9687,geoc_012453,17 QUARTIER peaksHART 93700 DRANCY,17 QUARTIER peaksHART,93700.0,DRANCY,QUARTIER,17,NaN,2.0,1.0,NaN,66.67,33.33,NaN,peaksHART,NaN,NaN
10493,10498,geoc_014949,37 RUE QUARTIER PARISION 94200 IVRY SUR SEINE,37 RUE QUARTIER PARISION,94200.0,IVRY SUR SEINE,RUE,37,QUARTIER,2.0,1.0,3,50.0,25.00,75.0,NaN,QUARTIER PARISION,AP
14349,14357,geoc_014238,00 QUARTIER AKIOUD IMMEUBLE D N22 99350 MARRAKECH,00 QUARTIER AKIOUD IMMEUBLE D N 22,99350.0,MARRAKECH,QUARTIER,00,NaN,2.0,1.0,NaN,28.57,14.29,NaN,AKIOUD IMMEUBLE D N 22,NaN,NaN
14618,14626,geoc_003664,MAISON FACHAO - QUARTIER VEDOKO 99327 COTOMOU,MAISON FACHAO QUARTIER VEDOKO,99327.0,COTOMOU,QUARTIER,NaN,"MAISON,QUARTIER",3.0,NaN,"1,3",75.0,NaN,"25.0,75.0",NaN,NaN,NaN
